# DSAB experiment 10: DeepSeek's answer with no reasoning

Asks DeepSeek-R1-Distill-Llama-8B True or False on every factual statement **with reasoning
switched off** (an empty think block, then `Answer:`), reads its Yes/No from one forward
pass, and compares that answer, item by item, with the plain probe from experiment 3.

The question it answers: on the items where the model's reasoning ends up wrong, the plain
probe still follows the true label (0.92 on F0). Did the model's own quick answer get those
items right (then the probe reads the model's belief before reasoning), or wrong too (then
the probe reads something the model never says)?

**Runtime:** A100 or L4. **Time:** about 25 minutes. **Results** go to the same Drive folder
as the first run, `MyDrive/dsab_review_results`, and a rerun after a disconnect skips
finished tasks.

## 1. Setup

In [ ]:
import os, torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> A100 or L4."
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{mem_gb:.0f} GB")
assert mem_gb >= 22, "An 8B model in fp16 needs an A100 or L4."

from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/dsab_review_results'
CACHE = '/content/dsab_cache'
os.makedirs(RESULTS, exist_ok=True)
os.environ['DSAB_RESULTS_DIR'] = RESULTS
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

REPO = '/content/Algoverse_Truth_Directions_Research'
if not os.path.exists(REPO):
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/boracandan/Algoverse_Truth_Directions_Research.git {REPO}
%cd {REPO}
!git checkout -q 60c8954 && git --no-pager log --oneline -1
!mkdir -p experiments/review_experiments

## 2. Write the code (run these, no need to read them)

In [ ]:
%%writefile experiments/review_experiments/common.py
"""
Shared helpers for the review experiments (exp3-exp9).

Everything here is lifted from the existing notebooks so results stay comparable with
results_database.csv:
  - extract_all_layer_activations is the Figure_3_Cross_Condition_Generalization version
    (length-sorted batches, left padding, final-token residual stream at every layer).
  - fit_linear_probe is the same bias-free, mean-centred probe (Adam, lr 1e-3,
    weight_decay 0.1, 1000 full-batch steps), with one change: it takes a seed.

What is new:
  - a seed on every probe, plus a bootstrap confidence interval on every AUROC. The seed
    only changes the probe's random init, and full-batch Adam lands in nearly the same
    place every time, so seed spread on its own will look tiny. The bootstrap over test
    items is the error bar that reflects how few test items there are (~290 for A1-A3).
  - a results file with more key columns (experiment, readout, seed, eval target, subset)
    so reruns with different seeds or readouts sit side by side instead of overwriting
    each other the way drop_duplicates(keep='last') does in results_database.csv.

GPU-only imports (torch, transformers) are deferred into the functions that need them so
the CPU experiments (exp8, exp9, and the data-prep halves of exp3/exp6) run on a laptop.
"""

import gc
import os
import re
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

REPO_ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = REPO_ROOT / "datasets"
# DSAB_RESULTS_DIR lets a Colab run write straight to Drive, so a disconnect loses nothing.
RESULTS_DIR = Path(os.environ.get("DSAB_RESULTS_DIR", Path(__file__).resolve().parent / "results"))
RESULTS_CSV = RESULTS_DIR / "review_results.csv"

TASKS = ["A1", "A2", "A3", "F0", "F1", "F2", "F3", "F4", "F5"]
ARITH_TASKS = ["A1", "A2", "A3"]
FACT_TASKS = ["F0", "F1", "F2", "F3", "F4", "F5"]

DEEPSEEK_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
DEEPSEEK_SHORT = "deepseek-r1-distill-8b"  # same label results_database.csv uses
DEEPSEEK_PAD_ID = 128001  # pad id hardcoded in the Figure 3 notebook

# Same folders as CONDITION_DICT in the existing notebooks.
CONDITION_DICT = {
    "no-prompt": DATA_DIR / "plain_dataset",
    "cot-zero-shot": DATA_DIR / "CoT_datasets" / "lexically_cleaned",
    "sentence-based-CoT": DATA_DIR / "CoT_datasets" / "sentence_based_lexically_cleaned",
    "no-prompt-chat-template": DATA_DIR / "plain_template_dataset",
    "ablation-filler-token": DATA_DIR / "ablation_datasets" / "filler_token_only",
    "ablation-instructions-and-template": DATA_DIR / "ablation_datasets" / "instructions_and_template",
    "ablation-instructions-only": DATA_DIR / "ablation_datasets" / "instructions_only",
}

DEEPSEEK_MARKERS = ["<｜begin▁of▁sentence｜>", "<｜User｜>"]


# ---------------------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------------------

def to_binary_labels(series):
    """Labels are stored as True/False in most files and 0/1 in others (F5). pandas reads
    True/False as bool already; strings are handled too in case a file was re-saved."""
    mapping = {"true": 1, "false": 0, "1": 1, "0": 0}
    if series.dtype == bool:
        return series.astype(int).to_numpy()
    return series.map(lambda v: mapping[str(v).strip().lower()]).astype(int).to_numpy()


def load_split(condition, task, split):
    path = CONDITION_DICT[condition] / f"{task}_{split}.csv"
    df = pd.read_csv(path)
    df["y"] = to_binary_labels(df["label"])
    return df


def statement_text(row):
    """The bare statement for any condition's row. Plain files have a `statement` column;
    every other file starts its text with the statement (after DeepSeek's BOS/User markers
    when a chat template was used) and follows it with a blank line or <｜Assistant｜>."""
    if "statement" in row and isinstance(row["statement"], str):
        return row["statement"].strip()
    text = row.get("generated_statement_texts", row.get("extracted_statement_texts"))
    for marker in DEEPSEEK_MARKERS:
        text = text.replace(marker, "")
    return re.split(r"\n\n|<｜Assistant｜>", text, maxsplit=1)[0].strip()


def ids_for(df, tokenizer=None):
    """Token-id lists for each row. Plain files carry no ids, so they are tokenized here
    exactly as the Figure 3 notebook does (tokenizer default, which adds BOS)."""
    if "extracted_statement_ids" in df.columns:
        return [literal_eval(x) if isinstance(x, str) else list(x) for x in df["extracted_statement_ids"]]
    if tokenizer is None:
        raise ValueError("plain dataset rows need a tokenizer to produce ids")
    return tokenizer(df["statement"].tolist())["input_ids"]


# ---------------------------------------------------------------------------------------
# Model + activations (GPU)
# ---------------------------------------------------------------------------------------

def load_model(model_id=DEEPSEEK_ID):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    kwargs = dict(device_map="cuda" if torch.cuda.is_available() else "cpu", trust_remote_code=True)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, **kwargs)
    except TypeError:  # transformers < 4.56 calls it torch_dtype
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, **kwargs)
    model.eval()
    return model, tokenizer


def extract_all_layer_activations(id_lists, model, pad_token_id, batch_size=4, desc="Extracting"):
    """Final-token residual stream at every layer, one array per layer, rows in input order.

    Same algorithm as the Figure 3 notebook: sort by length, left-pad, read [:, -1, :],
    un-sort. Readout position is controlled by the caller truncating each id list, which
    is exact for a causal model: the activation at position k only sees tokens 0..k, so
    reading the last token of ids[:k+1] equals reading position k of the full sequence.
    """
    import torch
    from tqdm.auto import tqdm

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    tensors = [torch.tensor(ids, dtype=torch.long) for ids in id_lists]
    lengths = torch.tensor([len(t) for t in tensors])
    sort_idx = torch.argsort(lengths)
    inverse_idx = torch.argsort(sort_idx)
    sorted_tensors = [tensors[i] for i in sort_idx.tolist()]

    n_layers = model.config.num_hidden_layers + 1
    per_layer = [[] for _ in range(n_layers)]
    device = next(model.parameters()).device

    with tqdm(total=len(sorted_tensors), desc=desc, leave=False) as pbar:
        for i in range(0, len(sorted_tensors), batch_size):
            batch = sorted_tensors[i:i + batch_size]
            padded = torch.nn.utils.rnn.pad_sequence(
                batch, batch_first=True, padding_value=pad_token_id, padding_side="left"
            ).to(device)
            # Mask by length, not by `padded != pad_token_id`: the pad id can also be a real
            # token inside a sequence (Granite reuses EOS as pad).
            lens = torch.tensor([len(t) for t in batch], device=device)
            positions = torch.arange(padded.shape[1], device=device).unsqueeze(0)
            attention_mask = (positions >= (padded.shape[1] - lens).unsqueeze(1)).long()
            with torch.no_grad():
                out = model(input_ids=padded, attention_mask=attention_mask, output_hidden_states=True)
            for layer_idx, hidden in enumerate(out.hidden_states):
                per_layer[layer_idx].append(hidden[:, -1, :].float().cpu())
            del out, padded, attention_mask
            pbar.update(len(batch))

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return [torch.cat(acts, dim=0)[inverse_idx].numpy() for acts in per_layer]


def cached_activations(cache_dir, key, compute_fn):
    """Optional disk cache (float16 .npz) keyed by a string the caller builds from model,
    condition, task, split and readout. Several experiments read the same activations
    (plain and sentence-based-CoT at the final token), so pointing them all at one cache
    folder on Drive saves most of the GPU time after the first run."""
    if cache_dir is None:
        return compute_fn()
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / f"{key}.npz"
    if path.exists():
        data = np.load(path)
        return [data[f"l{i}"].astype(np.float32) for i in range(len(data.files))]
    acts = compute_fn()
    np.savez(path, **{f"l{i}": a.astype(np.float16) for i, a in enumerate(acts)})
    return acts


# ---------------------------------------------------------------------------------------
# Probes + scoring
# ---------------------------------------------------------------------------------------

def fit_linear_probe(X_train, y_train, seed=0):
    """Bias-free logistic probe on mean-centred activations (as in the notebooks), seeded."""
    import torch
    import torch.nn as nn

    torch.manual_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    X = torch.tensor(X_train, dtype=torch.float32)
    train_mean = X.mean(dim=0)
    Xc = (X - train_mean).to(device)
    y = torch.tensor(y_train, dtype=torch.float32, device=device)

    probe = nn.Linear(X.shape[1], 1, bias=False).to(device)
    opt = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(1000):
        opt.zero_grad()
        loss_fn(probe(Xc).squeeze(-1), y).backward()
        opt.step()
    return probe.weight.detach().cpu().numpy().flatten(), train_mean.numpy()


def probe_scores(w, train_mean, X):
    return (X - train_mean) @ w


def _auroc_rows(y, scores):
    """AUROC for every row of 2-D (n_boot, n) arrays at once, via the Mann-Whitney rank
    formula with average ranks for ties (the same value roc_auc_score gives)."""
    ranks = rankdata(scores, axis=1)
    n_pos = y.sum(axis=1)
    n_neg = y.shape[1] - n_pos
    with np.errstate(invalid="ignore", divide="ignore"):
        return ((ranks * y).sum(axis=1) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def auroc_with_ci(y, scores, n_boot=1000, seed=0):
    """AUROC plus a 95% percentile bootstrap interval over test items. Returns NaNs when
    the subset has a single class (AUROC undefined) instead of the 0.5 the notebooks use,
    so an undefined number can never be read as chance. The bootstrap is vectorized: a
    loop of 1000 roc_auc_score calls per row made exp3 alone take about an hour of CPU."""
    y = np.asarray(y).astype(float)
    scores = np.asarray(scores, dtype=float)
    if len(np.unique(y)) < 2:
        return float("nan"), float("nan"), float("nan")
    point = roc_auc_score(y, scores)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(y), (n_boot, len(y)))
    boots = _auroc_rows(y[idx], scores[idx])
    boots = boots[np.isfinite(boots)]  # resamples that drew a single class
    if not len(boots):
        return float(point), float("nan"), float("nan")
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(point), float(lo), float(hi)


def parse_layers(arg, n_layers):
    """--layers all | 8,12,16 | 0-32:4"""
    if arg in (None, "all"):
        return list(range(n_layers))
    if ":" in arg:
        span, step = arg.split(":")
        lo, hi = span.split("-")
        return list(range(int(lo), min(int(hi), n_layers - 1) + 1, int(step)))
    return [int(x) for x in arg.split(",") if int(x) < n_layers]


# ---------------------------------------------------------------------------------------
# Results file
# ---------------------------------------------------------------------------------------

RESULT_KEYS = [
    "experiment", "model", "train_task", "test_task", "train_condition", "test_condition",
    "readout", "layer", "seed", "eval_target", "subset",
]
RESULT_COLUMNS = RESULT_KEYS + ["n_train", "n_test", "auroc", "ci_low", "ci_high"]


def append_results(rows, csv_path=None):
    """Append rows, replacing only rows that match on every key column (so a rerun of the
    same seed and readout replaces itself, and a new seed adds rows)."""
    if not rows:
        return
    csv_path = Path(csv_path or RESULTS_CSV)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    new = pd.DataFrame(rows)[RESULT_COLUMNS]
    if csv_path.exists():
        old = pd.read_csv(csv_path)
        new = pd.concat([old, new], ignore_index=True).drop_duplicates(subset=RESULT_KEYS, keep="last")
    new.to_csv(csv_path, index=False)


def already_done(experiment, **match):
    """True when the results file already has rows for this experiment and every
    column=value in `match`. Lets a rerun after a Colab disconnect skip finished work."""
    if not Path(RESULTS_CSV).exists():
        return False
    df = pd.read_csv(RESULTS_CSV)
    mask = df["experiment"] == experiment
    for col, val in match.items():
        mask &= df[col].astype(str) == str(val)
    return bool(mask.any())


def result_row(**kw):
    row = {c: kw.get(c, "") for c in RESULT_COLUMNS}
    row.update({k: v for k, v in kw.items() if k in RESULT_COLUMNS})
    return row

In [ ]:
%%writefile experiments/review_experiments/exp3_probe_vs_verdict.py
"""
Experiment 3: does the probe read the truth, or the verdict the model is about to give?

The sentence-based CoT prefixes stop right before the model's first verdict word. By then
the model has usually decided, so a probe there could be reading "truth" or "the answer I
am about to write". On most items the two agree (the model is right ~90-99% of the time),
so ordinary AUROC cannot tell them apart. They only come apart on the items where the
model's final answer is WRONG.

On those disagreement items, AUROC against the true label is the test:
  > 0.5  the probe sides with the truth, against the model's own answer
  < 0.5  the probe sides with the model's (wrong) answer
On that subset the verdict is exactly 1 - label, so AUROC vs verdict = 1 - AUROC vs label
and only the label number is reported.

Step 1 (CPU, no model): parse the model's answer out of every generated trace and write
datasets/CoT_datasets/verdicts/{task}.csv. Run with --parse-only on a laptop.

Step 2 (GPU): 5-fold cross-validation over train+test pooled, so every item gets an
out-of-fold score and all disagreement items are usable (the test split alone has ~50 in
F0, and almost none in A1-A3). Grouped by statement so a duplicated statement never sits
on both sides of a fold. Three probes per fold and layer:
  cot_label    trained on CoT activations with the true label      (the paper's probe)
  cot_verdict  trained on CoT activations with the model's answer  (a verdict probe)
  plain_label  trained on plain-statement activations, true label  (the transfer probe),
               scored both on plain activations and on CoT activations of held-out items

Arithmetic has almost no model errors, so this experiment can only settle the factual
tasks. For arithmetic, use exp4 (earlier readout points) instead.

Usage:
  python experiments/review_experiments/exp3_probe_vs_verdict.py --parse-only
  python experiments/review_experiments/exp3_probe_vs_verdict.py --tasks F0 F3 F4 F5 \
      --layers all --cache-dir /content/drive/MyDrive/dsab_cache
"""

import argparse
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

sys.path.insert(0, str(Path(__file__).resolve().parent))
import common as C  # noqa: E402

COT = "sentence-based-CoT"
PLAIN = "no-prompt"
VERDICT_DIR = C.DATA_DIR / "CoT_datasets" / "verdicts"


# ---------------------------------------------------------------------------------------
# Step 1: parse verdicts
# ---------------------------------------------------------------------------------------

def parse_verdict(generated_text):
    """Returns (final_answer, think_verdict) as 1/0/None.

    The prompt itself contains the template lines 'Answer: Yes' / 'Answer: No' and
    'The statement is true/false', so only text after <｜Assistant｜> is searched. The
    final 'Answer:' line is the model's committed answer. The think verdict only counts
    the exact line the prompt asks for ('The statement is true.' on its own line): a
    looser 'statement is true' match picks up hypotheticals mid-reasoning ('if X, the
    statement is true') and produced dozens of false conflicts on F3/F4.
    """
    gen = generated_text.split("<｜Assistant｜>", 1)[-1]
    answers = re.findall(r"Answer:\s*\**\s*(Yes|No)\b", gen, flags=re.IGNORECASE)
    thinks = re.findall(r"^\s*\**the statement is\s+(true|false)\.?\**\s*$", gen,
                        flags=re.IGNORECASE | re.MULTILINE)
    final = None if not answers else int(answers[-1].lower() == "yes")
    think = None if not thinks else int(thinks[-1].lower() == "true")
    return final, think


def build_verdicts(tasks):
    VERDICT_DIR.mkdir(parents=True, exist_ok=True)
    summary = []
    for task in tasks:
        parts = []
        for split in ["train", "test"]:
            df = C.load_split(COT, task, split)
            parsed = df["generated_statement_texts"].map(parse_verdict)
            out = pd.DataFrame({
                "split": split,
                "row": np.arange(len(df)),
                "statement": [C.statement_text(r) for _, r in df.iterrows()],
                "label": df["y"],
                "answer_verdict": [p[0] for p in parsed],
                "think_verdict": [p[1] for p in parsed],
            })
            # Committed answer first; fall back to the in-thinking line if the answer
            # line is missing (e.g. generation cut off before </think>).
            out["model_verdict"] = out["answer_verdict"].fillna(out["think_verdict"])
            out["verdict_source"] = np.where(out["answer_verdict"].notna(), "answer_line",
                                             np.where(out["think_verdict"].notna(), "think_line", "none"))
            parts.append(out)
        out = pd.concat(parts, ignore_index=True)
        out.to_csv(VERDICT_DIR / f"{task}.csv", index=False)

        has = out["model_verdict"].notna()
        both = out["answer_verdict"].notna() & out["think_verdict"].notna()
        summary.append({
            "task": task,
            "items": len(out),
            "parsed": int(has.sum()),
            "model_accuracy": round(float((out.loc[has, "model_verdict"] == out.loc[has, "label"]).mean()), 3),
            "disagree_total": int((has & (out["model_verdict"] != out["label"])).sum()),
            "disagree_test_split": int((has & (out["model_verdict"] != out["label"]) & (out["split"] == "test")).sum()),
            "think_vs_answer_conflicts": int((both & (out["answer_verdict"] != out["think_verdict"])).sum()),
        })
    summary = pd.DataFrame(summary)
    print(summary.to_string(index=False))
    C.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    summary.to_csv(C.RESULTS_DIR / "exp3_verdict_summary.csv", index=False)
    return summary


# ---------------------------------------------------------------------------------------
# Step 2: probes
# ---------------------------------------------------------------------------------------

def plain_rows_for(statements, task, tokenizer):
    """Plain-statement ids for the same statements, in the same order (exp6 matching)."""
    # Read the full plain files directly: this is a lookup table, never subsampled.
    plain = pd.concat([pd.read_csv(C.CONDITION_DICT[PLAIN] / f"{task}_{s}.csv") for s in ["train", "test"]],
                      ignore_index=True)
    lookup = dict(zip(plain["statement"].str.strip(), plain["statement"]))
    missing = [s for s in statements if s not in lookup]
    if missing:
        raise ValueError(f"{task}: {len(missing)} CoT statements have no plain match, e.g. {missing[0]!r}")
    return tokenizer([lookup[s] for s in statements])["input_ids"]


def run_probes(tasks, layers_arg, seeds, folds, cache_dir, batch_size):
    model, tokenizer = C.load_model(C.DEEPSEEK_ID)
    n_layers = model.config.num_hidden_layers + 1
    layers = C.parse_layers(layers_arg, n_layers)

    for task in tasks:
        todo = [s for s in seeds if not C.already_done("exp3_probe_vs_verdict", train_task=task, seed=s)]
        if not todo:
            print(f"\n{task}: already done, skipping (delete its rows from the results file to redo)")
            summarize(task)
            continue
        v = pd.read_csv(VERDICT_DIR / f"{task}.csv")
        cot = pd.concat([C.load_split(COT, task, s) for s in ["train", "test"]], ignore_index=True)
        assert len(cot) == len(v), "verdict file out of sync with CoT data; rerun --parse-only"

        keep = v["model_verdict"].notna().to_numpy()
        v, cot = v[keep].reset_index(drop=True), cot[keep].reset_index(drop=True)
        y = v["label"].to_numpy().astype(int)
        verdict = v["model_verdict"].to_numpy().astype(int)
        disagree = verdict != y
        print(f"\n{task}: {len(v)} items, {disagree.sum()} where the model's answer is wrong")

        cot_ids = C.ids_for(cot)
        plain_ids = plain_rows_for(v["statement"].tolist(), task, tokenizer)
        acts_cot = C.cached_activations(
            cache_dir, f"{C.DEEPSEEK_SHORT}__{COT}__{task}__all__final",
            lambda: C.extract_all_layer_activations(cot_ids, model, C.DEEPSEEK_PAD_ID, batch_size, f"{task} CoT"))
        acts_plain = C.cached_activations(
            cache_dir, f"{C.DEEPSEEK_SHORT}__{PLAIN}-matched__{task}__all__final",
            lambda: C.extract_all_layer_activations(plain_ids, model, C.DEEPSEEK_PAD_ID, batch_size, f"{task} plain"))

        groups = pd.factorize(v["statement"])[0]
        rows = []
        for seed in todo:
            # Stratify on label x verdict so each fold gets its share of the rare
            # disagreement items.
            strat = y * 2 + verdict
            splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
            fold_of = np.empty(len(v), dtype=int)
            for k, (_, te) in enumerate(splitter.split(np.zeros(len(v)), strat, groups)):
                fold_of[te] = k

            for layer in layers:
                Xc, Xp = acts_cot[layer], acts_plain[layer]
                oof = {name: np.zeros(len(v)) for name in
                       ["cot_label", "cot_verdict", "plain_label_on_plain", "plain_label_on_cot"]}
                for k in range(folds):
                    tr, te = fold_of != k, fold_of == k
                    w, m = C.fit_linear_probe(Xc[tr], y[tr], seed)
                    oof["cot_label"][te] = C.probe_scores(w, m, Xc[te])
                    w, m = C.fit_linear_probe(Xc[tr], verdict[tr], seed)
                    oof["cot_verdict"][te] = C.probe_scores(w, m, Xc[te])
                    w, m = C.fit_linear_probe(Xp[tr], y[tr], seed)
                    oof["plain_label_on_plain"][te] = C.probe_scores(w, m, Xp[te])
                    oof["plain_label_on_cot"][te] = C.probe_scores(w, m, Xc[te])

                for name, scores in oof.items():
                    train_cond = PLAIN if name.startswith("plain") else COT
                    test_cond = PLAIN if name.endswith("on_plain") else COT
                    for subset, mask in [("all", np.ones(len(v), bool)),
                                         ("model_wrong", disagree),
                                         ("model_right", ~disagree)]:
                        for target, labels in [("label", y), ("verdict", verdict)]:
                            if subset == "model_wrong" and target == "verdict":
                                continue  # = 1 - (vs label) on this subset
                            auc, lo, hi = C.auroc_with_ci(labels[mask], scores[mask], seed=seed)
                            rows.append(C.result_row(
                                experiment="exp3_probe_vs_verdict", model=C.DEEPSEEK_SHORT,
                                train_task=task, test_task=task,
                                train_condition=f"{train_cond}[{'verdict' if name == 'cot_verdict' else 'label'}]",
                                test_condition=test_cond, readout="final", layer=layer, seed=seed,
                                eval_target=target, subset=subset,
                                n_train=int(round(len(v) * (folds - 1) / folds)), n_test=int(mask.sum()),
                                auroc=auc, ci_low=lo, ci_high=hi))
            C.append_results(rows)
            rows = []
        summarize(task)


def summarize(task):
    """Per probe: choose the layer on ALL items (best AUROC vs the true label), then report
    the model-wrong AUROC at that layer. Choosing the layer on the model-wrong items
    themselves (e.g. the layer furthest from 0.5) would overstate the result."""
    df = pd.read_csv(C.RESULTS_CSV)
    df = df[(df["experiment"] == "exp3_probe_vs_verdict") & (df["train_task"] == task) & (df["eval_target"] == "label")]
    if df.empty:
        return
    print(f"\n{task}: AUROC vs TRUE label on items the model got wrong "
          f"(>0.5 = follows truth, <0.5 = follows the model's answer); layer chosen on all items")
    for (tr, te), g in df.groupby(["train_condition", "test_condition"]):
        g = g.groupby(["subset", "layer"])[["auroc", "ci_low", "ci_high", "n_test"]].mean()
        layer = g.loc["all", "auroc"].idxmax()
        a, w = g.loc[("all", layer)], g.loc[("model_wrong", layer)]
        print(f"  {tr:>30} -> {te:<20} layer {layer:>2}: all items {a.auroc:.3f} | "
              f"model wrong {w.auroc:.3f} [{w.ci_low:.3f}, {w.ci_high:.3f}] n={int(w.n_test)}")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--tasks", nargs="+", default=C.TASKS)
    ap.add_argument("--parse-only", action="store_true", help="only build the verdict files (CPU)")
    ap.add_argument("--layers", default="all", help="all | 8,16,24 | 0-32:4")
    ap.add_argument("--seeds", nargs="+", type=int, default=[0])
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--cache-dir", default=None)
    ap.add_argument("--batch-size", type=int, default=4)
    args = ap.parse_args()

    build_verdicts(args.tasks)
    if not args.parse_only:
        run_probes(args.tasks, args.layers, args.seeds, args.folds, args.cache_dir, args.batch_size)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile experiments/review_experiments/exp10_direct_answer.py
"""
Experiment 10: the model's answer with NO reasoning, item by item, next to the plain probe.

exp3 showed that on items where the model's post-reasoning answer is wrong, the plain probe
(read on the bare statement) still follows the true label (0.92 on F0). exp3 never asked
the model for an answer without reasoning, so it cannot say whether that probe reads the
model's own pre-reasoning belief or something the model never outputs. This collects that
answer and splits the items accordingly.

The no-reasoning answer: the statement goes in DeepSeek's chat template with Bora's
instruction from check_model_accuracy.ipynb, and the assistant turn is pre-filled with an empty think block and the answer
prefix (<think>\\n\\n</think>\\n\\nAnswer:), the usual way to switch reasoning off on the
R1 distills. One forward pass, then the next-token logits for " Yes" vs " No" give p_yes.
No generation. `yes_no_mass` (the probability on those two tokens) checks that the model is
answering in the asked format; a low value means the prompt is not working.

Small models answering without reasoning lean hard towards Yes (the 1.5B distill said Yes to
98% of F0 statements in a local test), and a raw argmax answer then mostly measures that
lean. So the script reports two answers: `direct_raw` (argmax) and `direct` (calibrated:
Yes for the items with the highest p_yes, as many as there are true statements in the task),
plus the AUROC of p_yes against the label, which no lean can change. The item splits use the
calibrated answer.

Per task the script then trains the plain probe exactly as exp3 does (same pooled items,
same 5 folds, same seed, layer chosen on all items) and reports, against the TRUE label:
  direct_wrong                plain probe on items the no-reasoning answer gets wrong
                              (> 0.5: probe follows the truth past the model's quick answer)
  cot_wrong & direct_right    items reasoning got wrong but the quick answer got right
  cot_wrong & direct_wrong    items both answers got wrong (> 0.5 here means the probe reads
                              something neither answer shows)
plus the no-reasoning accuracy per task and how often it agrees with the post-reasoning answer.

Usage:
  python experiments/review_experiments/exp10_direct_answer.py --tasks F0 F1 F2 F3 F4 F5 \
      --cache-dir /content/dsab_cache/exp10
"""

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

sys.path.insert(0, str(Path(__file__).resolve().parent))
import common as C  # noqa: E402
import exp3_probe_vs_verdict as E3  # noqa: E402

# Bora's EVAL_INSTRUCTIONS from validation_code/check_model_accuracy.ipynb, word for word,
# so DeepSeek's no-reasoning accuracy sits directly next to his Llama-3.1-8B numbers.
INSTRUCTION = (
    "Evaluate the statement for whether it is true or false. "
    "write exactly one line and nothing after it: "
    "\nAnswer: Yes or Answer: No"
)
NO_THINK_PREFILL = "\n</think>\n\nAnswer:"  # the template already ends with <think>\n


def direct_prompt_ids(statements, tokenizer):
    texts = [
        tokenizer.apply_chat_template([{"role": "user", "content": f"{s}\n\n{INSTRUCTION}"}],
                                      add_generation_prompt=True, tokenize=False) + NO_THINK_PREFILL
        for s in statements
    ]
    return [tokenizer(t, add_special_tokens=False)["input_ids"] for t in texts]


def direct_answers(id_lists, model, pad_id, yes_id, no_id, batch_size=16):
    """p(" Yes") and p(" No") at the next position, one forward pass per batch."""
    import torch
    from tqdm.auto import tqdm

    device = next(model.parameters()).device
    order = np.argsort([len(x) for x in id_lists])
    p_yes, p_no = np.zeros(len(id_lists)), np.zeros(len(id_lists))
    for i in tqdm(range(0, len(order), batch_size), desc="no-reasoning answers", leave=False):
        idx = order[i:i + batch_size]
        batch = [torch.tensor(id_lists[j]) for j in idx]
        padded = torch.nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=pad_id,
                                                 padding_side="left").to(device)
        lens = torch.tensor([len(t) for t in batch], device=device)
        pos = torch.arange(padded.shape[1], device=device).unsqueeze(0)
        mask = (pos >= (padded.shape[1] - lens).unsqueeze(1)).long()
        with torch.no_grad():
            probs = model(input_ids=padded, attention_mask=mask).logits[:, -1, :].float().softmax(-1)
        p_yes[idx] = probs[:, yes_id].cpu().numpy()
        p_no[idx] = probs[:, no_id].cpu().numpy()
    return p_yes, p_no


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--tasks", nargs="+", default=C.FACT_TASKS)
    ap.add_argument("--layers", default="all")
    ap.add_argument("--seed", type=int, default=0, help="same fold seed as exp3")
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--cache-dir", default=None)
    ap.add_argument("--batch-size", type=int, default=4)
    args = ap.parse_args()

    if not all((E3.VERDICT_DIR / f"{t}.csv").exists() for t in args.tasks):
        E3.build_verdicts(args.tasks)

    model, tokenizer = C.load_model(C.DEEPSEEK_ID)
    yes_id = tokenizer.encode(" Yes", add_special_tokens=False)[0]
    no_id = tokenizer.encode(" No", add_special_tokens=False)[0]
    layers = C.parse_layers(args.layers, model.config.num_hidden_layers + 1)
    out_dir = C.RESULTS_DIR / "exp10_direct_answers"
    out_dir.mkdir(parents=True, exist_ok=True)
    summary = []

    for task in args.tasks:
        if C.already_done("exp10_direct_answer", train_task=task):
            print(f"\n{task}: already done, skipping")
            continue
        v = pd.read_csv(E3.VERDICT_DIR / f"{task}.csv")
        v = v[v["model_verdict"].notna()].reset_index(drop=True)
        y = v["label"].to_numpy().astype(int)
        cot = v["model_verdict"].to_numpy().astype(int)

        # 1. no-reasoning answers
        p_yes, p_no = direct_answers(direct_prompt_ids(v["statement"].tolist(), tokenizer),
                                     model, C.DEEPSEEK_PAD_ID, yes_id, no_id)
        p_rel = p_yes / (p_yes + p_no)
        direct_raw = (p_yes > p_no).astype(int)
        # calibrated: as many Yes answers as there are true statements
        n_true = int(y.sum())
        direct = np.zeros(len(v), dtype=int)
        direct[np.argsort(-p_rel, kind="stable")[:n_true]] = 1
        direct_auc, d_lo, d_hi = C.auroc_with_ci(y, p_rel, seed=args.seed)
        items = v[["split", "statement", "label", "model_verdict"]].rename(columns={"model_verdict": "cot_answer"})
        items["direct_raw"], items["direct_answer"], items["p_yes"], items["yes_no_mass"] = direct_raw, direct, p_rel, p_yes + p_no
        items.to_csv(out_dir / f"{task}.csv", index=False)

        # 2. plain probe, out-of-fold, exactly as exp3
        plain_ids = E3.plain_rows_for(v["statement"].tolist(), task, tokenizer)
        acts = C.cached_activations(
            args.cache_dir, f"{C.DEEPSEEK_SHORT}__no-prompt-matched__{task}__all__final",
            lambda: C.extract_all_layer_activations(plain_ids, model, C.DEEPSEEK_PAD_ID, args.batch_size, f"{task} plain"))
        groups = pd.factorize(v["statement"])[0]
        splitter = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=args.seed)
        fold_of = np.empty(len(v), dtype=int)
        for k, (_, te) in enumerate(splitter.split(np.zeros(len(v)), y * 2 + cot, groups)):
            fold_of[te] = k

        subsets = {
            "all": np.ones(len(v), bool),
            "direct_wrong": direct != y,
            "cot_wrong": cot != y,
            "cot_wrong&direct_right": (cot != y) & (direct == y),
            "cot_wrong&direct_wrong": (cot != y) & (direct != y),
        }
        rows = []
        for layer in layers:
            scores = np.zeros(len(v))
            for k in range(args.folds):
                tr, te = fold_of != k, fold_of == k
                w, m = C.fit_linear_probe(acts[layer][tr], y[tr], args.seed)
                scores[te] = C.probe_scores(w, m, acts[layer][te])
            for name, mask in subsets.items():
                auc, lo, hi = C.auroc_with_ci(y[mask], scores[mask], seed=args.seed)
                rows.append(C.result_row(
                    experiment="exp10_direct_answer", model=C.DEEPSEEK_SHORT, train_task=task, test_task=task,
                    train_condition="no-prompt[label]", test_condition="no-prompt", readout="final", layer=layer,
                    seed=args.seed, eval_target="label", subset=name,
                    n_train=int(round(len(v) * (args.folds - 1) / args.folds)), n_test=int(mask.sum()),
                    auroc=auc, ci_low=lo, ci_high=hi))
        C.append_results(rows)

        df = pd.DataFrame(rows)
        layer = df[df["subset"] == "all"].set_index("layer")["auroc"].idxmax()
        at = df[df["layer"] == layer].set_index("subset")
        s = {
            "task": task, "items": len(v),
            "no_reasoning_yes_rate_raw": round(float(direct_raw.mean()), 3),
            "no_reasoning_accuracy_raw": round(float((direct_raw == y).mean()), 3),
            "no_reasoning_accuracy_calibrated": round(float((direct == y).mean()), 3),
            "no_reasoning_p_yes_AUROC": f"{direct_auc:.3f} [{d_lo:.2f}, {d_hi:.2f}]",
            "with_reasoning_accuracy": round(float((cot == y).mean()), 3),
            "answers_agree": round(float((direct == cot).mean()), 3),
            "median_yes_no_mass": round(float(np.median(p_yes + p_no)), 3),
            "probe_layer": int(layer),
        }
        for name in subsets:
            r = at.loc[name]
            s[f"{name} n"] = int(r.n_test)
            s[f"{name} AUROC"] = f"{r.auroc:.3f} [{r.ci_low:.2f}, {r.ci_high:.2f}]" if r.auroc == r.auroc else "n/a"
        summary.append(s)
        print(pd.Series(s).to_string(), "\n")

    if summary:
        path = C.RESULTS_DIR / "exp10_summary.csv"
        old = pd.read_csv(path) if path.exists() else pd.DataFrame()
        new = pd.concat([old, pd.DataFrame(summary)]).drop_duplicates("task", keep="last")
        new.to_csv(path, index=False)
        print(new.T.to_string())


if __name__ == "__main__":
    main()

## 3. Check the prompt works (1 minute)

Prints the model's next few tokens for five statements. Each should start with Yes or No.
If it starts thinking instead, stop and tell Archana before running the rest. All five
saying Yes is fine: models answering without reasoning lean towards Yes, and the script
corrects for that (see exp10_summary.csv, columns no_reasoning_yes_rate_raw and
no_reasoning_p_yes_AUROC).

In [ ]:
import sys; sys.path.insert(0, 'experiments/review_experiments')
import pandas as pd, common as C, exp3_probe_vs_verdict as E3, exp10_direct_answer as E10
E3.build_verdicts(['F0'])
model, tok = C.load_model(C.DEEPSEEK_ID)
v = pd.read_csv(E3.VERDICT_DIR / 'F0.csv').sample(5, random_state=0)
for s, ids in zip(v.statement, E10.direct_prompt_ids(v.statement.tolist(), tok)):
    out = model.generate(torch.tensor([ids]).to(model.device), max_new_tokens=5, do_sample=False)
    print(f"{s!r:70} -> {tok.decode(out[0][len(ids):])!r}")
del model; import gc; gc.collect(); torch.cuda.empty_cache()

## 4. Run it on the factual tasks (~20 min), then arithmetic (~5 min)

In [ ]:
!python experiments/review_experiments/exp10_direct_answer.py --tasks F0 F1 F2 F3 F4 F5 --cache-dir {CACHE}/exp10
!python experiments/review_experiments/exp10_direct_answer.py --tasks A1 A2 A3 --cache-dir {CACHE}/exp10
!rm -rf {CACHE}/exp10

## 5. Summary and a zip of the results

In [ ]:
s = pd.read_csv(f'{RESULTS}/exp10_summary.csv')
display(s.set_index('task').T)
!cd /content/drive/MyDrive && zip -qr /content/dsab_exp10_results.zip dsab_review_results/exp10_summary.csv dsab_review_results/exp10_direct_answers dsab_review_results/review_results.csv
from google.colab import files
files.download('/content/dsab_exp10_results.zip')